# Decoding

In [ ]:
import pandas as pd
import numpy as np
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from matplotlib.lines import Line2D
from sklearn.metrics import accuracy_score, f1_score
import sys
from pathlib import Path

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT.parent / 'LB'))
from src.decoding import cleandfdecodingTS

path = ROOT / 'ieeg_shortWOBS_fs250' / 'BJH026_info.json'
with open(path) as json_data:
    d = json.load(json_data)
    time = d['time_epoch']

modality = {'ieeg':'navy', 'meg':'orange'}


In [ ]:
# Clean each available decoding result once, immediately after loading the time vector.
RECOMPUTE_CLEANED_DECODING = False  # True overwrites existing caches after results/time settings change.
DECODING_RESULTS_DIR = ROOT / 'out' / 'decoding'

# Only full-length results: cropped results require their own matching time vector.
decoding_csv_files = sorted(DECODING_RESULTS_DIR.glob('*/*/fullmodel/*full.csv'))
cleaned_decoding_created = 0
cleaned_decoding_reused = 0
for decoding_csv_path in decoding_csv_files:
    cleaned_decoding_path = decoding_csv_path.with_name(decoding_csv_path.stem + '_cleaned.pkl')
    if cleaned_decoding_path.is_file() and not RECOMPUTE_CLEANED_DECODING:
        cleaned_decoding_reused += 1
        continue
    print(f'Cleaning {decoding_csv_path.relative_to(DECODING_RESULTS_DIR)}', flush=True)
    df_final = pd.read_csv(decoding_csv_path)
    cleaned_decoding_result = cleandfdecodingTS(df_final, len(time))
    # Preserve the complete dictionary, including NumPy arrays and None values.
    # Replace only after a successful write, so an interrupted write cannot become a cache hit.
    cleaned_decoding_temporary_path = cleaned_decoding_path.with_suffix('.pkl.tmp')
    pd.to_pickle(cleaned_decoding_result, cleaned_decoding_temporary_path)
    cleaned_decoding_temporary_path.replace(cleaned_decoding_path)
    cleaned_decoding_created += 1

print(f'Decoding caches: {cleaned_decoding_created} written, {cleaned_decoding_reused} already available.')
if not decoding_csv_files:
    print(f'No full-length decoding result CSVs found under {DECODING_RESULTS_DIR}. '
          'Run this cell again after generating or copying the results.')
# Plotting cells below load these files directly. Rerun this cell when new CSVs arrive.


In [8]:
band = 'broadband'
method_pca = 'mean'
data_aug_method = 'mean'
PC_use = 0
model_name = 'LR'
dim_red_method = 'PLSSVD'

if not os.path.isfile(f'out/decoding/ieeg/broadband/fullmodel/{dim_red_method}_{method_pca}_{data_aug_method}_{PC_use}_{model_name}full.csv'):
    print(f'submit_to_cluster -n 2 -q short.q "python vs4_1_decoding.py --pc_use {PC_use} --model {model_name} --meg-kind paired_coverage --iteration 100 --method {dim_red_method}"  --project MINDLAB2021_MEG-TempSeqAges')


In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(20, 10))
fig.suptitle(f'{model_name} -- Compo {PC_use+1} -- {dim_red_method}')

for d, dataset_ in enumerate(['ieeg', 'meg']):

    file = f'out/decoding/{dataset_}/broadband/fullmodel/{dim_red_method}_{method_pca}_{data_aug_method}_{PC_use}_{model_name}full_iteration0_time_courses.npz'
    with np.load(file,allow_pickle=False) as saved:
        train = saved['train_condition_mean']
   
    df_final = pd.read_csv(f'out/decoding/{dataset_}/broadband/fullmodel/{dim_red_method}_{method_pca}_{data_aug_method}_{PC_use}_{model_name}full.csv')
    result = pd.read_pickle(f'out/decoding/{dataset_}/broadband/fullmodel/{dim_red_method}_{method_pca}_{data_aug_method}_{PC_use}_{model_name}full_cleaned.pkl')
    weight_mean= np.abs(result['weight']).mean(0)
    weight_std= np.abs(result['weight']).std(0)
    weight_sh_mean= result['weight_sh_mean']
    weight_sh_std= result['weight_sh_std']

    ax[d].plot(time, weight_mean, color=modality[dataset_])
    ax[d].fill_between(time, weight_mean - weight_std,weight_mean+weight_std, alpha = 0.2, color = modality[dataset_])
    
    ax[d].plot(time, weight_sh_mean, color='black',alpha=0.5,  label = f'Mean {int(df_final.loc[0, "iter_perm"])}-perm model over {int(df_final.loc[0, "iter"])} iter')
    ax[d].fill_between(time, abs(weight_sh_mean), abs(weight_sh_mean) + weight_sh_std, alpha = 0.1, color = 'black', label = f'Sd {int(df_final.loc[0, "iter_perm"])}-perm model')
    #ax[d].plot(time[20:], - np.percentile(result['weight_sh'], 95, axis=0)[20:], alpha = 1,ls='--', linewidth=0.6, color = 'black')
    ax[d].plot(time, np.percentile(result['weight_sh'], 95, axis=0), alpha = 1, ls='--', linewidth=0.6, color = 'black', label = f'95% percentil {int(df_final.loc[0, "iter_perm"])}-perm model')
    ax[d].set_ylabel('Classifier weights')
    ax[d].grid()
    ax[d].set_title(dataset_ + ' - Accuracy: ' + str(100*np.round(result['acc'], 2)) + '%')
    axt = ax[d].twinx()
    axt.set_ylabel('Network activity')
    axt.plot(time, train[0, :], ls=':', lw = 1, color='blue', label = 'Network TS')
    axt.plot(time, train[1, :], ls=':', lw = 1, color='red')

In [ ]:
fig, ax=plt.subplots(figsize=(10, 1))

for d, dataset_ in enumerate(modality.keys()):
    result = pd.read_pickle(f'out/decoding/{dataset_}/{band}/fullmodel/{dim_red_method}_{method_pca}_{data_aug_method}_{PC_use}_{model_name}full_cleaned.pkl')
    weight_mean= np.abs(result['weight']).mean(0)
    weight_sh_95= np.percentile(result['weight_sh'], 95, axis=0)
    weight_thr = np.where(weight_mean>weight_sh_95, PC_use+ d*0.1, np.nan)
    ax.plot(time, weight_thr, color=modality[dataset_])
    ax.grid()

In [ ]:
# Big plot with all the accuracies per component and per models CLS and DIM RED
fig, ax = plt.subplots(1, 2, figsize=(10, 3), sharey=True)

method_dim_red = {'PCA': ':', 'PLSSVD':'--'}
method_clf = {'LR' : ':', 'RandomForest':'--', 'SVC_linear': '-.'}

for m, a in zip(method_dim_red.keys(), ax):
    a.set_title(m)
    for model_name in method_clf.keys():
        for dataset_ in modality.keys():
            acc =[]
            acc_s =[]
            for PC_use in range(4) : 

                result = pd.read_pickle(f'out/decoding/{dataset_}/{band}/fullmodel/{m}_{method_pca}_{data_aug_method}_{PC_use}_{model_name}full_cleaned.pkl')
                acc.append(result['acc'])
                acc_s.append(result['acc_sh'])
            a.plot(['PC' + str(i+1) for i in range(len(acc))], acc, ls=method_clf[model_name], color=modality[dataset_],  alpha = 1, lw=1.5, label=model_name + ' ' + dataset_)
            a.plot(['PC' + str(i+1) for i in range(len(acc))], acc_s,  ls=method_clf[model_name], color=modality[dataset_], alpha = 0.5, lw=1)

            a.grid()

In [ ]:
# heatmap of the accuracies compo x models for each methods
import seaborn as sns

fig, ax= plt.subplots(1, 2)
for a, dataset_ in zip(ax, modality.keys()):
    a.set_title(dataset_)
    lab = []
    values = np.zeros((4, 4))
    for iclf, model_name in enumerate(method_clf.keys()):
        for im, m in enumerate(method_dim_red.keys()):
            lab.append(model_name + ' ' + m)
            for PC_use in range(4) : 
                result = pd.read_pickle(f'out/decoding/{dataset_}/{band}/fullmodel/{m}_{method_pca}_{data_aug_method}_{PC_use}_{model_name}full_cleaned.pkl')
                values[im + iclf, PC_use]=result['acc']
    sns.heatmap(values, yticklabels=lab, ax=a, vmin=0.5, vmax=1, cmap='Reds', annot=True)


In [ ]:
# all the clf weights together 
model_name = 'LR'
m = 'PLSSVD'

fig, ax = plt.subplots(1, 2, figsize=(8, 4))
fig.suptitle(model_name)
for im, m in enumerate(method_dim_red.keys()) :
    for d, dataset_ in enumerate(modality.keys()):
        acc =[]
        acc_s =[]
        for PC_use in range(5) : 

            result = pd.read_pickle(f'out/decoding/{dataset_}/{band}/fullmodel/{m}_{method_pca}_{data_aug_method}_{PC_use}_{model_name}full_cleaned.pkl')
            weight_mean= np.abs(result['weight']).mean(0)
            weight_sh_95= np.percentile(result['weight_sh'], 95, axis=0)
            weight_thr = np.where(weight_mean>weight_sh_95, PC_use+ d*0.1, np.nan)

            ax[im].plot(time, weight_thr, color=modality[dataset_])
            ax[im].set_title(m)

for a in ax :       
    a.grid()
